In [2]:
# ── CELL 1a — Install only ────────────────────────────────────────────────────
import subprocess, sys

# P100 = sm_60. Last CUDA version supporting sm_60 is CUDA 11.8.
# torch 2.4.1+cu118 is the sweet spot: modern enough, still has sm_60 kernels.
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch==2.4.1+cu118",
    "torchvision==0.19.1+cu118",
    "torchaudio==2.4.1+cu118",
    "--index-url", "https://download.pytorch.org/whl/cu118"
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "timm", "albumentations", "seaborn", "opencv-python-headless"
])

print("✅ Installation complete.")
print(">>> NOW: run the kernel restart cell")
print(">>> THEN: run from CELL 1b downward")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.5/857.5 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 73.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 44.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 91.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 32.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.1/204.1 MB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# ── CELL 1a.5 — Restart kernel ────────────────────────────────────────────────
import IPython
IPython.get_ipython().kernel.do_shutdown(restart=True)


{'status': 'ok', 'restart': True}

In [1]:
# ── CELL 1b — Imports (run after kernel restart) ──────────────────────────────
import random, time, json, shutil, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import (Dataset, DataLoader, ConcatDataset,
                               WeightedRandomSampler, random_split)
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

print(f"PyTorch     : {torch.__version__}")
print(f"GPU         : {torch.cuda.get_device_name(0)}")
print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("✅ Imports complete")

PyTorch     : 2.4.1+cu118
GPU         : Tesla P100-PCIE-16GB
VRAM        : 17.1 GB
✅ Imports complete


In [2]:
class Config:
    # ── Dataset paths (update to match your Kaggle input slugs) ──────────────
    RAF_DB_DIR    = "/kaggle/input/datasets/prajwalm2213/dataset3/RAF-DB"
    FER2013_DIR   = "/kaggle/input/datasets/prajwalm2213/dataset/FER-2013"
    JAFFE_DIR     = "/kaggle/input/datasets/prajwalm2213/dataset2/jaffe/jaffe"
    CKP_DIR       = "/kaggle/input/datasets/prajwalm2213/dataset/CK"
    AFFECTNET_DIR = "/kaggle/input/datasets/prajwalm2213/dataset/Affectnet/archive (3)"
 
    # ── Output ────────────────────────────────────────────────────────────────
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    LOG_DIR        = "/kaggle/working/logs"
    EVAL_DIR       = "/kaggle/working/eval_results"
 
    # ── Model ─────────────────────────────────────────────────────────────────
    IMAGE_SIZE     = 224
    NUM_CLASSES    = 11          # compound emotions only
    EMBED_DIM      = 512
    LSTM_HIDDEN    = 256
    LSTM_LAYERS    = 2
    DROPOUT        = 0.3
    VIT_MODEL_NAME = "vit_base_patch16_224"
 
    # ── Training ──────────────────────────────────────────────────────────────
    BATCH_SIZE     = 32
    LEARNING_RATE  = 2e-4
    WEIGHT_DECAY   = 1e-4
    PATIENCE       = 10
    SEED           = 42
 
    # ── Compound class definitions ────────────────────────────────────────────
    COMPOUND_CLASSES = [
        "Happily Surprised",   "Happily Disgusted",  "Sadly Fearful",
        "Sadly Angry",         "Sadly Surprised",    "Sadly Disgusted",
        "Fearfully Angry",     "Fearfully Surprised","Angrily Surprised",
        "Angrily Disgusted",   "Disgustedly Surprised",
    ]
    BASIC_CLASSES = ["Angry","Disgust","Fear","Happy","Sad","Surprise","Neutral"]
 
    # compound_idx → (basic_emotion_1, basic_emotion_2)
    # indices align to BASIC_CLASSES above
    COMPOUND_TO_BASIC = {
        0:(3,5), 1:(3,1), 2:(4,2),  3:(4,0), 4:(4,5),
        5:(4,1), 6:(2,0), 7:(2,5),  8:(0,5), 9:(0,1), 10:(1,5),
    }
 
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
cfg = Config()
for d in [cfg.CHECKPOINT_DIR, cfg.LOG_DIR, cfg.EVAL_DIR]:
    os.makedirs(d, exist_ok=True)
 
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
 
set_seed(cfg.SEED)
print(f"Device : {cfg.DEVICE}")

Device : cuda


In [3]:
def get_train_transforms():
    return A.Compose([
        A.Resize(cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=20, p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=3, p=1.0),
            A.GaussNoise(p=1.0),
            A.MotionBlur(p=1.0),
        ], p=0.3),
        A.ColorJitter(brightness=0.3, contrast=0.3,
                      saturation=0.2, hue=0.1, p=0.5),
        A.CoarseDropout(num_holes_range=(1,8),
                        hole_height_range=(8,16),
                        hole_width_range=(8,16), p=0.3),
        A.Affine(translate_percent=0.1, scale=(0.85,1.15),
                 rotate=(-15,15), p=0.4),
        A.Normalize(mean=[0.485,0.456,0.406],
                    std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])
 
def get_val_transforms():
    return A.Compose([
        A.Resize(cfg.IMAGE_SIZE, cfg.IMAGE_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406],
                    std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

In [4]:
class FER2013Pool(Dataset):
    """Loads FER-2013 images without transforms — used as synthesis pool."""
    def __init__(self, split="train"):
        self.samples = []
        data_dir = os.path.join(cfg.FER2013_DIR, split)
        for label_idx, cls in enumerate(cfg.BASIC_CLASSES):
            class_dir = os.path.join(data_dir, cls.lower())
            if not os.path.exists(class_dir): continue
            for fname in os.listdir(class_dir):
                if fname.lower().endswith((".jpg",".jpeg",".png")):
                    self.samples.append((os.path.join(class_dir, fname),
                                         label_idx))
        # group by class for fast lookup
        self.by_class = {i: [] for i in range(7)}
        for path, label in self.samples:
            self.by_class[label].append(path)
        print(f"FER-2013 pool ({split}): {len(self.samples)} images")
        for i, name in enumerate(cfg.BASIC_CLASSES):
            print(f"  {name:<10}: {len(self.by_class[i])}")
 
 
# ── CK+ / AffectNet basic emotion pool ───────────────────────────────────────
FOLDER_TO_BASIC = {
    "anger":0,"angry":0,
    "disgust":1,"disgusted":1,
    "fear":2,"fearful":2,
    "happy":3,"happiness":3,
    "sad":4,"sadness":4,
    "surprise":5,"surprised":5,
    "neutral":6,
    # AffectNet numeric
    "0":6,"1":3,"2":4,"3":5,"4":2,"5":1,"6":0,
}
 
def collect_pool(root_dir, max_per_class=400):
    by_class = {i: [] for i in range(7)}
    if not os.path.exists(root_dir):
        print(f"  ⚠  Not found: {root_dir}")
        return by_class
    print(f"  Scanning: {root_dir}")
    subfolders = sorted(os.listdir(root_dir))
    print(f"  Found subfolders: {subfolders}")
    for folder in subfolders:
        idx = FOLDER_TO_BASIC.get(folder.lower().strip())
        if idx is None: continue
        class_dir = os.path.join(root_dir, folder)
        if not os.path.isdir(class_dir): continue
        paths = [os.path.join(class_dir, f)
                 for f in os.listdir(class_dir)
                 if f.lower().endswith((".jpg",".jpeg",".png",".bmp",".tiff"))]
        random.shuffle(paths)
        by_class[idx].extend(paths[:max_per_class])
        print(f"    -> Matched '{folder}' to class {idx} "
              f"({len(by_class[idx])} images)")
    total = sum(len(v) for v in by_class.values())
    print(f"  Total collected from {os.path.basename(root_dir)}: {total}")
    return by_class
 
 
# ── Core synthetic compound dataset ──────────────────────────────────────────
class SyntheticCompoundDataset(Dataset):
    """
    Morphological upper/lower face splice.
    Upper half from emotion_1, lower half from emotion_2.
    """
    def __init__(self, by_class, samples_per_class=800, transform=None,
                 label_offset=0):
        self.transform    = transform
        self.label_offset = label_offset   # always 0 — compound labels are 0–10
        self.samples      = self._generate(by_class, samples_per_class)
 
    def _generate(self, by_class, n):
        samples = []
        for compound_idx, (e1, e2) in cfg.COMPOUND_TO_BASIC.items():
            pool1 = by_class.get(e1, [])
            pool2 = by_class.get(e2, [])
            if len(pool1) < 5 or len(pool2) < 5:
                print(f"  ⚠  Compound {cfg.COMPOUND_CLASSES[compound_idx]}: "
                      f"insufficient images (e1={e1}:{len(pool1)}, "
                      f"e2={e2}:{len(pool2)})")
                continue
            for _ in range(n):
                samples.append((random.choice(pool1),
                                 random.choice(pool2),
                                 compound_idx))
        return samples
 
    def _splice(self, img1, img2):
        h = img1.shape[0]
        alpha   = random.uniform(0.45, 0.55)
        boundary = int(h * alpha)
        decay   = 10
        result  = img1.copy().astype(np.float32)
        result[boundary:] = img2[boundary:].astype(np.float32)
        for d in range(decay):
            row = boundary - decay // 2 + d
            if 0 <= row < h:
                t = d / decay
                result[row] = ((1-t)*img1[row].astype(np.float32)
                               + t*img2[row].astype(np.float32))
        return np.clip(result, 0, 255).astype(np.uint8)
 
    def _load(self, path):
        img = cv2.imread(path)
        if img is None:
            img = np.array(Image.open(path).convert("RGB"))
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return cv2.resize(img, (cfg.IMAGE_SIZE, cfg.IMAGE_SIZE))
 
    def __len__(self): return len(self.samples)
 
    def __getitem__(self, idx):
        p1, p2, label = self.samples[idx]
        mixed = self._splice(self._load(p1), self._load(p2))
        if self.transform:
            mixed = self.transform(image=mixed)["image"]
        return mixed, torch.tensor(label, dtype=torch.long)
 
 
# ── Build all dataloaders ─────────────────────────────────────────────────────
def get_dataloaders():
    print("\n── Building compound emotion dataloaders ──")
 
    # Source pools
    fer_pool = FER2013Pool(split="train")
    ckp_pool = collect_pool(cfg.CKP_DIR,       max_per_class=300)
    aff_pool = collect_pool(cfg.AFFECTNET_DIR,  max_per_class=300)
 
    # Merge pools: FER-2013 (primary) + CK+ + AffectNet
    merged_pool = {i: [] for i in range(7)}
    for i in range(7):
        merged_pool[i] = (fer_pool.by_class[i]
                          + ckp_pool[i]
                          + aff_pool[i])
    print("\nMerged pool sizes:")
    for i, name in enumerate(cfg.BASIC_CLASSES):
        print(f"  {name:<10}: {len(merged_pool[i])}")
 
    # ── Full synthetic compound dataset ──────────────────────────────────────
    # 900 samples per class × 11 classes = 9,900 total
    # 80% train / 20% val  →  7,920 train / 1,980 val
    full_dataset = SyntheticCompoundDataset(
        merged_pool,
        samples_per_class=900,
        transform=None,         # transforms applied per-split below
    )
    total   = len(full_dataset)
    n_val   = int(0.20 * total)
    n_train = total - n_val
 
    train_idx, val_idx = random_split(
        range(total), [n_train, n_val],
        generator=torch.Generator().manual_seed(cfg.SEED)
    )
 
    # Wrap with transforms
    class TransformSubset(Dataset):
        def __init__(self, base_dataset, indices, transform):
            self.base      = base_dataset
            self.indices   = list(indices)
            self.transform = transform
        def __len__(self): return len(self.indices)
        def __getitem__(self, i):
            p1, p2, label = self.base.samples[self.indices[i]]
            mixed = self.base._splice(
                self.base._load(p1),
                self.base._load(p2)
            )
            if self.transform:
                mixed = self.transform(image=mixed)["image"]
            return mixed, torch.tensor(label, dtype=torch.long)
 
    train_set = TransformSubset(full_dataset, train_idx, get_train_transforms())
    val_set   = TransformSubset(full_dataset, val_idx,   get_val_transforms())
 
    print(f"\nDataset split:")
    print(f"  Train : {len(train_set)} compound samples")
    print(f"  Val   : {len(val_set)}   compound samples  ← used for early stopping")
 
    # ── Balanced sampler (equal weight per compound class) ───────────────────
    train_labels = [full_dataset.samples[i][2] for i in train_idx]
    counts  = np.bincount(train_labels, minlength=cfg.NUM_CLASSES).astype(np.float32)
    weights = 1.0 / (counts + 1e-6)
    sw = torch.tensor([weights[l] for l in train_labels], dtype=torch.float)
    sampler = WeightedRandomSampler(sw, len(sw), replacement=True)
 
    train_loader = DataLoader(train_set, batch_size=cfg.BATCH_SIZE,
                              sampler=sampler, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=cfg.BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)
 
    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val   batches: {len(val_loader)}")
    return train_loader, val_loader

In [5]:
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        b, c, _, _ = x.shape
        avg = self.fc(self.avg_pool(x).view(b, c))
        mx  = self.fc(self.max_pool(x).view(b, c))
        return x * self.sigmoid(avg + mx).view(b, c, 1, 1)
 
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size,
                                 padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return x * self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
 
class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention()
    def forward(self, x):
        return self.sa(self.ca(x))
 
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, stride=stride,
                                  padding=1, bias=False)
        self.bn1      = nn.BatchNorm2d(out_ch)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2      = nn.BatchNorm2d(out_ch)
        self.cbam     = CBAMBlock(out_ch)
        self.relu     = nn.ReLU(inplace=True)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch))
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.cbam(out)
        return self.relu(out + self.shortcut(x))
 
class SACNN(nn.Module):
    def __init__(self, embed_dim=512, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1))
        self.stage1 = self._make_stage(64,  128, stride=1, blocks=2)
        self.stage2 = self._make_stage(128, 256, stride=2, blocks=2)
        self.stage3 = self._make_stage(256, 512, stride=2, blocks=2)
        self.stage4 = self._make_stage(512, 512, stride=2, blocks=1)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.drop   = nn.Dropout(dropout)
        self.fc     = nn.Linear(512, embed_dim)
        self.bn_out = nn.BatchNorm1d(embed_dim)
        self._init_weights()
    def _make_stage(self, in_ch, out_ch, stride, blocks):
        layers = [ResidualBlock(in_ch, out_ch, stride=stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_ch, out_ch))
        return nn.Sequential(*layers)
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out",
                                        nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x); x = self.stage2(x)
        x = self.stage3(x); x = self.stage4(x)
        x = self.pool(x).flatten(1)
        x = self.drop(x)
        return self.bn_out(self.fc(x))
 
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_ch=3, patch_dim=256):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_ch, patch_dim, kernel_size=patch_size,
                      stride=patch_size),
            nn.Flatten(2))
    def forward(self, x):
        return self.proj(x).transpose(1, 2)
 
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1, bias=False))
    def forward(self, lstm_out):
        scores  = self.attn(lstm_out)
        weights = F.softmax(scores, dim=1)
        return (weights * lstm_out).sum(dim=1), weights.squeeze(-1)
 
class ALSTM(nn.Module):
    def __init__(self, img_size=224, patch_size=16, patch_dim=256,
                 hidden_dim=256, num_layers=2, embed_dim=512, dropout=0.3):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size,
                                          patch_dim=patch_dim)
        self.lstm = nn.LSTM(patch_dim, hidden_dim, num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.attention = TemporalAttention(hidden_dim)
        self.drop   = nn.Dropout(dropout)
        self.fc     = nn.Linear(hidden_dim * 2, embed_dim)
        self.bn_out = nn.BatchNorm1d(embed_dim)
        self._init_weights()
    def _init_weights(self):
        for name, param in self.lstm.named_parameters():
            if "weight_ih" in name: nn.init.xavier_uniform_(param)
            elif "weight_hh" in name: nn.init.orthogonal_(param)
            elif "bias" in name: nn.init.zeros_(param)
        nn.init.xavier_normal_(self.fc.weight)
    def forward(self, x):
        patches     = self.patch_embed(x)
        lstm_out, _ = self.lstm(patches)
        context, _  = self.attention(lstm_out)
        return self.bn_out(self.fc(self.drop(context)))
 
class ViTEncoder(nn.Module):
    def __init__(self, model_name="vit_base_patch16_224",
                 embed_dim=512, dropout=0.3, pretrained=True):
        super().__init__()
        self.vit = timm.create_model(model_name, pretrained=pretrained,
                                     num_classes=0)
        vit_dim  = self.vit.num_features
        self.projection = nn.Sequential(
            nn.LayerNorm(vit_dim),
            nn.Dropout(dropout),
            nn.Linear(vit_dim, embed_dim),
            nn.GELU(),
            nn.BatchNorm1d(embed_dim))
        self._freeze_all()
    def _freeze_all(self):
        for p in self.vit.parameters(): p.requires_grad = False
    def unfreeze_last_n_blocks(self, n=8):
        for p in self.vit.norm.parameters(): p.requires_grad = True
        for block in list(self.vit.blocks)[-n:]:
            for p in block.parameters(): p.requires_grad = True
        tr = sum(p.numel() for p in self.vit.parameters() if p.requires_grad)
        print(f"  ViT unfrozen last {n} blocks — trainable ViT params: {tr:,}")
    def unfreeze_all(self):
        for p in self.vit.parameters(): p.requires_grad = True
        print("  ViT fully unfrozen")
    def forward(self, x):
        return self.projection(self.vit(x))
 
class AttentionFusion(nn.Module):
    def __init__(self, embed_dim=512):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(embed_dim * 3, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 3),
            nn.Softmax(dim=-1))
    def forward(self, f1, f2, f3):
        combined = torch.cat([f1, f2, f3], dim=-1)
        weights  = self.gate(combined)
        fused = (weights[:,0:1]*f1 +
                 weights[:,1:2]*f2 +
                 weights[:,2:3]*f3)
        return fused, weights
 
class CompoundEmotionModel(nn.Module):
    def __init__(self, num_classes=11, embed_dim=512, dropout=0.3):
        super().__init__()
        self.sacnn  = SACNN(embed_dim=embed_dim, dropout=dropout)
        self.alstm  = ALSTM(hidden_dim=cfg.LSTM_HIDDEN,
                            num_layers=cfg.LSTM_LAYERS,
                            embed_dim=embed_dim, dropout=dropout)
        self.vit    = ViTEncoder(model_name=cfg.VIT_MODEL_NAME,
                                 embed_dim=embed_dim, dropout=dropout,
                                 pretrained=True)
        self.fusion = AttentionFusion(embed_dim=embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 512),
            nn.LayerNorm(512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes))
        self._init_classifier()
 
    def _init_classifier(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
 
    # ── Stage setup helpers ───────────────────────────────────────────────────
    def stage1_setup(self):
        """Freeze entire ViT. Train SA-CNN + ALSTM + fusion + classifier."""
        self.vit._freeze_all()
        tr = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Stage 1 trainable params: {tr:,}")
 
    def stage2_setup(self, n=8):
        """Unfreeze last n ViT blocks."""
        self.vit.unfreeze_last_n_blocks(n)
        tr = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Stage 2 trainable params: {tr:,}")
 
    def stage3_setup(self):
        """Unfreeze full ViT."""
        self.vit.unfreeze_all()
        tr = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Stage 3 trainable params: {tr:,}")
 
    def forward(self, x):
        f1 = self.sacnn(x)
        f2 = self.alstm(x)
        f3 = self.vit(x)
        fused, weights = self.fusion(f1, f2, f3)
        return self.classifier(fused), weights
 
print("Model architecture loaded ✓")

Model architecture loaded ✓


In [6]:
def mixup_batch(imgs, labels, num_classes, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    mixed = lam * imgs + (1 - lam) * imgs[idx]
    y_a = F.one_hot(labels, num_classes).float()
    y_b = F.one_hot(labels[idx], num_classes).float()
    return mixed, lam * y_a + (1 - lam) * y_b
 
def compute_metrics(preds, labels):
    p, l = np.array(preds), np.array(labels)
    acc = (p == l).mean() * 100
    f1  = f1_score(l, p, average="weighted", zero_division=0) * 100
    return acc, f1
 
def train_one_epoch(model, loader, optimizer, scheduler,
                    scaler, criterion, epoch, total):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
 
    for imgs, labels in loader:
        imgs   = imgs.to(cfg.DEVICE, non_blocking=True)
        labels = labels.to(cfg.DEVICE, non_blocking=True)
 
        use_mixup = random.random() < 0.4
        if use_mixup:
            imgs, soft_labels = mixup_batch(imgs, labels, cfg.NUM_CLASSES)
 
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            logits, _ = model(imgs)
            if use_mixup:
                lp   = F.log_softmax(logits, dim=-1)
                loss = -(soft_labels * lp).sum(dim=-1).mean()
            else:
                loss = criterion(logits, labels)
 
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
 
        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
 
    acc, f1 = compute_metrics(all_preds, all_labels)
    return total_loss / len(loader), acc, f1
 
@torch.no_grad()
def validate(model, loader, criterion, epoch, total):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels, fw_log = [], [], []
 
    for imgs, labels in loader:
        imgs   = imgs.to(cfg.DEVICE, non_blocking=True)
        labels = labels.to(cfg.DEVICE, non_blocking=True)
        with torch.amp.autocast("cuda"):
            logits, weights = model(imgs)
            loss = criterion(logits, labels)
        total_loss  += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        fw_log.append(weights.cpu().numpy())
 
    fw = np.concatenate(fw_log, axis=0).mean(axis=0)
    acc, f1 = compute_metrics(all_preds, all_labels)
    return total_loss / len(loader), acc, f1, fw
 
def save_ckpt(state, fname):
    torch.save(state, os.path.join(cfg.CHECKPOINT_DIR, fname))
    print(f"    Saved → {fname}")
 
def load_ckpt(model, fname):
    path = os.path.join(cfg.CHECKPOINT_DIR, fname)
    if not os.path.exists(path):
        print(f"  ⚠  {fname} not found")
        return 0, 0.0
    ckpt = torch.load(path, map_location=cfg.DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model"])
    print(f"  Loaded {fname} — compound_acc={ckpt['best_acc']:.2f}%")
    return ckpt["epoch"], ckpt["best_acc"]
 
def run_stage(model, train_loader, val_loader,
              stage, epochs, lr, ckpt_name):
    """
    Single training stage.
    Val loader is ALWAYS the compound val split — early stopping
    is based on compound emotion accuracy.
    """
    # Class-balanced CE with label smoothing
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=cfg.WEIGHT_DECAY
    )
    scheduler = OneCycleLR(
        optimizer, max_lr=lr,
        steps_per_epoch=len(train_loader),
        epochs=epochs, pct_start=0.3,
        anneal_strategy="cos",
        div_factor=10, final_div_factor=100
    )
    scaler      = torch.amp.GradScaler("cuda")
    best_acc    = 0.0
    patience_ct = 0
    log         = []
 
    print(f"\n{'='*65}")
    print(f"  Stage {stage} — {epochs} epochs | max_lr={lr:.2e}")
    print(f"  Val = compound emotion split (this is the correct metric)")
    print(f"{'='*65}")
 
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc, tr_f1 = train_one_epoch(
            model, train_loader, optimizer, scheduler,
            scaler, criterion, epoch, epochs)
        vl_loss, vl_acc, vl_f1, fw = validate(
            model, val_loader, criterion, epoch, epochs)
 
        elapsed = time.time() - t0
        print(f"  E{epoch:02d}/{epochs} | "
              f"tr_acc={tr_acc:.1f}% f1={tr_f1:.1f}% | "
              f"val_acc={vl_acc:.1f}% f1={vl_f1:.1f}% | "
              f"fw=[{fw[0]:.2f},{fw[1]:.2f},{fw[2]:.2f}] | "
              f"{elapsed:.0f}s")
 
        log.append({"epoch": epoch, "stage": stage,
                    "tr_acc": tr_acc, "vl_acc": vl_acc, "vl_f1": vl_f1})
 
        if vl_acc > best_acc:
            best_acc    = vl_acc
            patience_ct = 0
            save_ckpt({"epoch": epoch, "model": model.state_dict(),
                       "optimizer": optimizer.state_dict(),
                       "best_acc": best_acc,
                       "stage": stage}, ckpt_name)
        else:
            patience_ct += 1
            if patience_ct >= cfg.PATIENCE:
                print(f"  Early stop at epoch {epoch} "
                      f"(no improvement for {cfg.PATIENCE} epochs)")
                break
 
    print(f"\n  Stage {stage} best compound accuracy: {best_acc:.2f}%")
    return log, best_acc
 
print("Training utilities loaded ✓")

Training utilities loaded ✓


In [7]:
set_seed(cfg.SEED)
 
# ── Dataloaders ───────────────────────────────────────────────────────────────
train_loader, val_loader = get_dataloaders()
 
# ── Model ─────────────────────────────────────────────────────────────────────
model = CompoundEmotionModel(
    num_classes=cfg.NUM_CLASSES,
    embed_dim=cfg.EMBED_DIM,
    dropout=cfg.DROPOUT,
).to(cfg.DEVICE)
 
total_p = sum(p.numel() for p in model.parameters())
print(f"\nTotal model parameters: {total_p:,}")
 
# ── Stage 1: ViT frozen — train SA-CNN + ALSTM + fusion ──────────────────────
model.stage1_setup()
log1, best1 = run_stage(
    model, train_loader, val_loader,
    stage=1, epochs=25, lr=cfg.LEARNING_RATE,
    ckpt_name="stage1_best.pth"
)
load_ckpt(model, "stage1_best.pth")
torch.cuda.empty_cache()
 
# ── Stage 2: Unfreeze last 8 ViT blocks ───────────────────────────────────────
model.stage2_setup(n=8)
log2, best2 = run_stage(
    model, train_loader, val_loader,
    stage=2, epochs=25, lr=cfg.LEARNING_RATE * 0.2,
    ckpt_name="stage2_best.pth"
)
load_ckpt(model, "stage2_best.pth")
torch.cuda.empty_cache()
 
# ── Stage 3: Full ViT unfreeze — low LR fine-tune ────────────────────────────
model.stage3_setup()
model.vit.vit.set_grad_checkpointing(True)   # saves VRAM for full ViT
log3, best3 = run_stage(
    model, train_loader, val_loader,
    stage=3, epochs=10, lr=cfg.LEARNING_RATE * 0.02,
    ckpt_name="stage3_best.pth"
)
 
# ── Save training log ─────────────────────────────────────────────────────────
full_log = log1 + log2 + log3
with open(os.path.join(cfg.LOG_DIR, "training_log.json"), "w") as f:
    json.dump(full_log, f, indent=2)
 
print(f"\n{'='*65}")
print(f"  TRAINING COMPLETE")
print(f"  Stage 1 best: {best1:.2f}%")
print(f"  Stage 2 best: {best2:.2f}%")
print(f"  Stage 3 best: {best3:.2f}%")
print(f"{'='*65}")
 
# Copy best checkpoint to working root for easy download
shutil.copy2(os.path.join(cfg.CHECKPOINT_DIR, "stage3_best.pth"),
             "/kaggle/working/stage3_best.pth")
print("✓ Best checkpoint → /kaggle/working/stage3_best.pth")


── Building compound emotion dataloaders ──
FER-2013 pool (train): 28709 images
  Angry     : 3995
  Disgust   : 436
  Fear      : 4097
  Happy     : 7215
  Sad       : 4830
  Surprise  : 3171
  Neutral   : 4965
  Scanning: /kaggle/input/datasets/prajwalm2213/dataset/CK
  Found subfolders: ['anger', 'contempt', 'disgust', 'fear', 'happy', 'sadness', 'surprise']
    -> Matched 'anger' to class 0 (135 images)
    -> Matched 'disgust' to class 1 (177 images)
    -> Matched 'fear' to class 2 (75 images)
    -> Matched 'happy' to class 3 (207 images)
    -> Matched 'sadness' to class 4 (84 images)
    -> Matched 'surprise' to class 5 (249 images)
  Total collected from CK: 927
  Scanning: /kaggle/input/datasets/prajwalm2213/dataset/Affectnet/archive (3)
  Found subfolders: ['Test', 'Train', 'labels.csv']
  Total collected from archive (3): 0

Merged pool sizes:
  Angry     : 4130
  Disgust   : 613
  Fear      : 4172
  Happy     : 7422
  Sad       : 4914
  Surprise  : 3420
  Neutral   : 496


Total model parameters: 106,407,676
  Stage 1 trainable params: 20,609,020

  Stage 1 — 25 epochs | max_lr=2.00e-04
  Val = compound emotion split (this is the correct metric)
  E01/25 | tr_acc=10.2% f1=9.8% | val_acc=14.5% f1=13.3% | fw=[0.30,0.42,0.28] | 107s
    Saved → stage1_best.pth
  E02/25 | tr_acc=11.7% f1=11.7% | val_acc=17.7% f1=16.7% | fw=[0.27,0.38,0.36] | 105s
    Saved → stage1_best.pth
  E03/25 | tr_acc=13.4% f1=13.3% | val_acc=21.7% f1=20.0% | fw=[0.29,0.21,0.50] | 105s
    Saved → stage1_best.pth
  E04/25 | tr_acc=15.3% f1=15.2% | val_acc=22.1% f1=21.4% | fw=[0.29,0.27,0.44] | 105s
    Saved → stage1_best.pth
  E05/25 | tr_acc=17.8% f1=17.6% | val_acc=23.2% f1=20.6% | fw=[0.18,0.13,0.69] | 105s
    Saved → stage1_best.pth
  E06/25 | tr_acc=19.7% f1=19.4% | val_acc=24.8% f1=23.7% | fw=[0.17,0.08,0.75] | 105s
    Saved → stage1_best.pth
  E07/25 | tr_acc=20.3% f1=20.1% | val_acc=26.8% f1=25.6% | fw=[0.10,0.09,0.81] | 105s
    Saved → stage1_best.pth
  E08/25 | tr_acc=2

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


  E01/10 | tr_acc=81.3% f1=81.3% | val_acc=62.5% f1=62.3% | fw=[0.10,0.02,0.88] | 230s
    Saved → stage3_best.pth


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


KeyboardInterrupt: 

In [8]:
# ── Download stage2_best.pth ──────────────────────────────────────────────────
import shutil, os
from IPython.display import FileLink

src  = os.path.join(cfg.CHECKPOINT_DIR, "stage2_best.pth")
dest = "/kaggle/working/stage2_best.pth"

shutil.copy2(src, dest)
print(f"✅ Copied: {dest}")
print(f"   Size  : {os.path.getsize(dest)/1e6:.1f} MB")
FileLink(dest)

✅ Copied: /kaggle/working/stage2_best.pth
   Size  : 1044.5 MB


/kaggle/working/stage2_best.pth

In [7]:
# ── CELL 7 — Resume from Stage 3 only ────────────────────────────────────────
set_seed(cfg.SEED)

# ── Dataloaders ───────────────────────────────────────────────────────────────
train_loader, val_loader = get_dataloaders()

# ── Model ─────────────────────────────────────────────────────────────────────
model = CompoundEmotionModel(
    num_classes=cfg.NUM_CLASSES,
    embed_dim=cfg.EMBED_DIM,
    dropout=cfg.DROPOUT,
).to(cfg.DEVICE)

# ── Load stage2 checkpoint from uploaded dataset ──────────────────────────────
# Update this path to match wherever you uploaded the .pth file in Kaggle
STAGE2_CKPT = "/kaggle/input/models/prajwalm2213/stage-2/pytorch/default/1/stage2_best.pth"

ckpt = torch.load(STAGE2_CKPT, map_location=cfg.DEVICE, weights_only=False)
model.load_state_dict(ckpt["model"])
print(f"✅ Loaded stage2_best.pth — best_acc={ckpt['best_acc']:.2f}%")

# ── Stage 3: Full ViT unfreeze — low LR fine-tune ────────────────────────────
model.stage3_setup()
model.vit.vit.set_grad_checkpointing(True)
log3, best3 = run_stage(
    model, train_loader, val_loader,
    stage=3, epochs=10, lr=cfg.LEARNING_RATE * 0.02,
    ckpt_name="stage3_best.pth"
)

# ── Save training log ─────────────────────────────────────────────────────────
with open(os.path.join(cfg.LOG_DIR, "training_log_stage3.json"), "w") as f:
    json.dump(log3, f, indent=2)

print(f"\n{'='*65}")
print(f"  STAGE 3 COMPLETE")
print(f"  Stage 3 best: {best3:.2f}%")
print(f"{'='*65}")

shutil.copy2(os.path.join(cfg.CHECKPOINT_DIR, "stage3_best.pth"),
             "/kaggle/working/stage3_best.pth")
print("✅ Best checkpoint → /kaggle/working/stage3_best.pth")


── Building compound emotion dataloaders ──
FER-2013 pool (train): 28709 images
  Angry     : 3995
  Disgust   : 436
  Fear      : 4097
  Happy     : 7215
  Sad       : 4830
  Surprise  : 3171
  Neutral   : 4965
  Scanning: /kaggle/input/datasets/prajwalm2213/dataset/CK
  Found subfolders: ['anger', 'contempt', 'disgust', 'fear', 'happy', 'sadness', 'surprise']
    -> Matched 'anger' to class 0 (135 images)
    -> Matched 'disgust' to class 1 (177 images)
    -> Matched 'fear' to class 2 (75 images)
    -> Matched 'happy' to class 3 (207 images)
    -> Matched 'sadness' to class 4 (84 images)
    -> Matched 'surprise' to class 5 (249 images)
  Total collected from CK: 927
  Scanning: /kaggle/input/datasets/prajwalm2213/dataset/Affectnet/archive (3)
  Found subfolders: ['Test', 'Train', 'labels.csv']
  Total collected from archive (3): 0

Merged pool sizes:
  Angry     : 4130
  Disgust   : 613
  Fear      : 4172
  Happy     : 7422
  Sad       : 4914
  Surprise  : 3420
  Neutral   : 496

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

✅ Loaded stage2_best.pth — best_acc=62.88%
  ViT fully unfrozen
  Stage 3 trainable params: 106,407,676

  Stage 3 — 10 epochs | max_lr=4.00e-06
  Val = compound emotion split (this is the correct metric)


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


  E01/10 | tr_acc=77.8% f1=77.8% | val_acc=62.7% f1=62.4% | fw=[0.09,0.02,0.89] | 231s
    Saved → stage3_best.pth


...

  E08/10 | tr_acc=77.8% f1=77.8% | val_acc=63.6% f1=63.4% | fw=[0.09,0.02,0.89] | 229s


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


  E09/10 | tr_acc=75.7% f1=75.7% | val_acc=64.3% f1=64.0% | fw=[0.08,0.02,0.90] | 229s
    Saved → stage3_best.pth


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


  E10/10 | tr_acc=74.7% f1=74.7% | val_acc=63.1% f1=62.9% | fw=[0.09,0.02,0.90] | 229s

  Stage 3 best compound accuracy: 64.34%

  STAGE 3 COMPLETE
  Stage 3 best: 64.34%
✅ Best checkpoint → /kaggle/working/stage3_best.pth


In [8]:
import shutil, os
from IPython.display import FileLink

src  = os.path.join(cfg.CHECKPOINT_DIR, "stage3_best.pth")
dest = "/kaggle/working/stage3_best.pth"
shutil.copy2(src, dest)
print(f"✅ Copied: {dest}")
print(f"   Size  : {os.path.getsize(dest)/1e6:.1f} MB")
FileLink(dest)

✅ Copied: /kaggle/working/stage3_best.pth
   Size  : 1277.3 MB


/kaggle/working/stage3_best.pth

In [9]:
import os
print(os.listdir("/kaggle/input/models/prajwalm2213/stage-2/pytorch/default/1"))

['stage2_best.pth']


In [10]:
def full_evaluation(model, val_loader, ckpt_name="stage3_best.pth"):
    load_ckpt(model, ckpt_name)
    model.eval()
 
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs = imgs.to(cfg.DEVICE)
            logits, _ = model(imgs)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    preds  = np.array(all_preds)
    labels = np.array(all_labels)
 
    acc   = accuracy_score(labels, preds) * 100
    prec  = f1_score(labels, preds, average="weighted", zero_division=0) * 100
    f1_w  = f1_score(labels, preds, average="weighted", zero_division=0) * 100
    f1_m  = f1_score(labels, preds, average="macro",    zero_division=0) * 100
 
    print(f"\n{'='*65}")
    print(f"  FINAL COMPOUND EMOTION RESULTS")
    print(f"{'='*65}")
    print(f"  Accuracy       : {acc:.2f}%")
    print(f"  Weighted F1    : {f1_w:.2f}%")
    print(f"  Macro F1       : {f1_m:.2f}%")
 
    report = classification_report(
        labels, preds,
        target_names=cfg.COMPOUND_CLASSES,
        labels=list(range(11)),
        digits=4, zero_division=0,
    )
    print(f"\n{report}")
 
    # ── Per-class accuracy ────────────────────────────────────────────────────
    cm = confusion_matrix(labels, preds, labels=list(range(11)))
    pca = cm.diagonal() / (cm.sum(axis=1) + 1e-9) * 100
    print("  Per-class Accuracy:")
    for name, a in zip(cfg.COMPOUND_CLASSES, pca):
        bar = "█" * int(a / 5)
        print(f"    {name:<25} {a:6.2f}%  {bar}")
 
    # ── Confusion matrix plot ─────────────────────────────────────────────────
    short = [n.replace("ily","").replace("fully","").replace("rily","")
             for n in cfg.COMPOUND_CLASSES]
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=short, yticklabels=short,
                linewidths=0.5, ax=ax)
    ax.set_title("Confusion Matrix — Compound Emotion Recognition",
                 fontsize=14, fontweight="bold", pad=15)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.tight_layout()
    cm_path = os.path.join(cfg.EVAL_DIR, "confusion_matrix_final.png")
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  Confusion matrix saved → {cm_path}")
 
    # ── Training curve ────────────────────────────────────────────────────────
    with open(os.path.join(cfg.LOG_DIR, "training_log.json")) as f:
        log = json.load(f)
    epochs_  = [e["epoch"] + (e["stage"]-1)*25 for e in log]
    val_accs = [e["vl_acc"] for e in log]
 
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(val_accs, color="#2E86AB", linewidth=2, label="Val Compound Acc")
    ax.axhline(max(val_accs), color="#E84855", linestyle="--",
               linewidth=1, label=f"Best: {max(val_accs):.2f}%")
    ax.set_xlabel("Epoch (cumulative across stages)")
    ax.set_ylabel("Compound Accuracy (%)")
    ax.set_title("Training Curve — Compound Emotion Recognition",
                 fontweight="bold")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    curve_path = os.path.join(cfg.EVAL_DIR, "training_curve.png")
    plt.savefig(curve_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Training curve  saved → {curve_path}")
 
    # ── Save JSON ─────────────────────────────────────────────────────────────
    results = {
        "accuracy": round(acc, 2),
        "weighted_f1": round(f1_w, 2),
        "macro_f1": round(f1_m, 2),
        "per_class_accuracy": {
            name: round(float(a), 2)
            for name, a in zip(cfg.COMPOUND_CLASSES, pca)
        },
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
    }
    out = os.path.join(cfg.EVAL_DIR, "final_results.json")
    with open(out, "w") as f:
        json.dump(results, f, indent=2)
    print(f"  Results JSON    saved → {out}")
    print("\n  Download all files from Kaggle Output tab.")
    return results
 
 
results = full_evaluation(model, val_loader, ckpt_name="stage3_best.pth")
 

  Loaded stage3_best.pth — compound_acc=64.34%

  FINAL COMPOUND EMOTION RESULTS
  Accuracy       : 63.84%
  Weighted F1    : 63.52%
  Macro F1       : 63.11%

                       precision    recall  f1-score   support

    Happily Surprised     0.6588    0.6474    0.6531       173
    Happily Disgusted     0.7487    0.7606    0.7546       188
        Sadly Fearful     0.6000    0.4263    0.4985       190
          Sadly Angry     0.5143    0.5806    0.5455       186
      Sadly Surprised     0.5749    0.5134    0.5424       187
      Sadly Disgusted     0.7102    0.6757    0.6925       185
      Fearfully Angry     0.6062    0.6500    0.6273       180
  Fearfully Surprised     0.4682    0.5548    0.5078       146
    Angrily Surprised     0.5676    0.5060    0.5350       166
    Angrily Disgusted     0.6888    0.7297    0.7087       185
Disgustedly Surprised     0.8235    0.9381    0.8771       194

             accuracy                         0.6384      1980
            macro a

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/logs/training_log.json'

In [11]:
# ── Per-Dataset Evaluation Cell ───────────────────────────────────────────────
# Evaluates stage3_best.pth on each source dataset separately:
#   JAFFE, CK+, RAF-DB, AffectNet, FER-2013
#
# How it works:
#   Each dataset contains BASIC emotion images.
#   We synthesise compound images on-the-fly (same splice logic as training)
#   and evaluate the compound classifier on them.
#   This tells you how well each data source supports compound recognition.
# ─────────────────────────────────────────────────────────────────────────────

import os, random, json
import numpy as np
import torch
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns


# ── JAFFE pool ────────────────────────────────────────────────────────────────
# JAFFE folders are named like "AN", "DI", "FE", "HA", "NE", "SA", "SU"
JAFFE_FOLDER_MAP = {
    "an": 0, "di": 1, "fe": 2, "ha": 3,
    "ne": 6, "sa": 4, "su": 5,
}

def collect_jaffe_pool(root):
    """
    JAFFE images are PGM files named like 'KA.AN1.39.pgm'.
    The emotion code is the 2nd dot-segment (AN, DI, FE, HA, NE, SA, SU).
    """
    by_class = {i: [] for i in range(7)}
    if not os.path.exists(root):
        print(f"  ⚠  JAFFE not found: {root}")
        return by_class

    # Try subfolder structure first
    subfolders = [d for d in os.listdir(root)
                  if os.path.isdir(os.path.join(root, d))]
    if subfolders:
        print(f"  JAFFE subfolders found: {subfolders}")
        for folder in subfolders:
            idx = JAFFE_FOLDER_MAP.get(folder.lower()[:2])
            if idx is None:
                continue
            class_dir = os.path.join(root, folder)
            paths = [os.path.join(class_dir, f)
                     for f in os.listdir(class_dir)
                     if f.lower().endswith((".pgm", ".jpg", ".jpeg", ".png"))]
            by_class[idx].extend(paths)
            print(f"    {folder} → class {idx} ({cfg.BASIC_CLASSES[idx]}): "
                  f"{len(paths)} imgs")
    else:
        # Flat directory — parse emotion from filename
        print(f"  JAFFE flat directory, parsing filenames...")
        for fname in os.listdir(root):
            if not fname.lower().endswith((".pgm", ".jpg", ".jpeg", ".png")):
                continue
            parts = fname.split(".")
            if len(parts) >= 2:
                emo_code = parts[1][:2].lower()
                idx = JAFFE_FOLDER_MAP.get(emo_code)
                if idx is not None:
                    by_class[idx].append(os.path.join(root, fname))

    total = sum(len(v) for v in by_class.values())
    print(f"  JAFFE total: {total} images")
    for i, name in enumerate(cfg.BASIC_CLASSES):
        if by_class[i]:
            print(f"    {name:<10}: {len(by_class[i])}")
    return by_class


# ── Synthetic eval dataset (same splice as training) ─────────────────────────
class SyntheticEvalDataset(Dataset):
    """
    Builds N compound samples per class from a basic-emotion pool.
    Uses the same upper/lower splice as SyntheticCompoundDataset in training.
    """
    def __init__(self, by_class, samples_per_class=100, transform=None):
        self.transform = transform
        self.samples   = self._generate(by_class, samples_per_class)

    def _generate(self, by_class, n):
        samples = []
        for compound_idx, (e1, e2) in cfg.COMPOUND_TO_BASIC.items():
            pool1 = by_class.get(e1, [])
            pool2 = by_class.get(e2, [])
            if len(pool1) < 2 or len(pool2) < 2:
                print(f"    ⚠  {cfg.COMPOUND_CLASSES[compound_idx]}: "
                      f"skipped (e1={cfg.BASIC_CLASSES[e1]}:{len(pool1)}, "
                      f"e2={cfg.BASIC_CLASSES[e2]}:{len(pool2)})")
                continue
            actual_n = min(n, len(pool1), len(pool2))
            for _ in range(actual_n):
                samples.append((random.choice(pool1),
                                 random.choice(pool2),
                                 compound_idx))
        return samples

    def _load(self, path):
        img = cv2.imread(path)
        if img is None:
            img = np.array(Image.open(path).convert("RGB"))
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return cv2.resize(img, (cfg.IMAGE_SIZE, cfg.IMAGE_SIZE))

    def _splice(self, img1, img2):
        h     = img1.shape[0]
        alpha = random.uniform(0.45, 0.55)
        boundary = int(h * alpha)
        decay = 10
        result = img1.copy().astype(np.float32)
        result[boundary:] = img2[boundary:].astype(np.float32)
        for d in range(decay):
            row = boundary - decay // 2 + d
            if 0 <= row < h:
                t = d / decay
                result[row] = ((1 - t) * img1[row].astype(np.float32)
                               + t * img2[row].astype(np.float32))
        return np.clip(result, 0, 255).astype(np.uint8)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p1, p2, label = self.samples[idx]
        mixed = self._splice(self._load(p1), self._load(p2))
        if self.transform:
            mixed = self.transform(image=mixed)["image"]
        return mixed, torch.tensor(label, dtype=torch.long)


# ── Evaluate one dataset ──────────────────────────────────────────────────────
def evaluate_dataset(model, by_class, dataset_name,
                     samples_per_class=100, batch_size=32):
    """
    Synthesises compound images from `by_class` pool and runs inference.
    Returns a results dict.
    """
    print(f"\n{'─'*60}")
    print(f"  Evaluating: {dataset_name}")
    print(f"{'─'*60}")

    eval_ds = SyntheticEvalDataset(
        by_class,
        samples_per_class=samples_per_class,
        transform=get_val_transforms()
    )

    if len(eval_ds) == 0:
        print(f"  ✗ No samples generated for {dataset_name} — skipping.")
        return None

    print(f"  Generated {len(eval_ds)} synthetic compound samples")

    loader = DataLoader(eval_ds, batch_size=batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(cfg.DEVICE)
            logits, _ = model(imgs)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    preds  = np.array(all_preds)
    labels = np.array(all_labels)

    # Only evaluate classes that actually have samples
    present_classes = sorted(set(labels.tolist()))
    present_names   = [cfg.COMPOUND_CLASSES[i] for i in present_classes]

    acc  = accuracy_score(labels, preds) * 100
    f1_w = f1_score(labels, preds, average="weighted", zero_division=0) * 100
    f1_m = f1_score(labels, preds, average="macro",    zero_division=0) * 100

    print(f"\n  {'Metric':<20} {'Value':>8}")
    print(f"  {'─'*30}")
    print(f"  {'Accuracy':<20} {acc:>7.2f}%")
    print(f"  {'Weighted F1':<20} {f1_w:>7.2f}%")
    print(f"  {'Macro F1':<20} {f1_m:>7.2f}%")
    print(f"  {'Samples evaluated':<20} {len(preds):>8}")

    report = classification_report(
        labels, preds,
        target_names=present_names,
        labels=present_classes,
        digits=4, zero_division=0
    )
    print(f"\n{report}")

    # Per-class accuracy
    cm  = confusion_matrix(labels, preds, labels=present_classes)
    pca = cm.diagonal() / (cm.sum(axis=1) + 1e-9) * 100
    print("  Per-class Accuracy:")
    for name, a in zip(present_names, pca):
        bar = "█" * int(a / 5)
        print(f"    {name:<25} {a:6.2f}%  {bar}")

    # Confusion matrix plot
    short = [n.replace("ily", "").replace("fully", "").replace("rily", "")
             for n in present_names]
    fig, ax = plt.subplots(figsize=(max(8, len(present_classes)),
                                    max(6, len(present_classes) - 1)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=short, yticklabels=short,
                linewidths=0.5, ax=ax)
    ax.set_title(f"Confusion Matrix — {dataset_name}",
                 fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.tight_layout()
    safe_name = dataset_name.replace(" ", "_").replace("/", "-").replace("+", "plus")
    cm_path   = os.path.join(cfg.EVAL_DIR, f"cm_{safe_name}.png")
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  Confusion matrix → {cm_path}")

    return {
        "dataset":          dataset_name,
        "n_samples":        int(len(preds)),
        "accuracy":         round(acc, 2),
        "weighted_f1":      round(f1_w, 2),
        "macro_f1":         round(f1_m, 2),
        "per_class_accuracy": {
            name: round(float(a), 2)
            for name, a in zip(present_names, pca)
        },
        "present_classes":  present_classes,
    }


# ── Build all per-dataset pools ───────────────────────────────────────────────
def build_all_pools():
    pools = {}

    # 1. JAFFE
    pools["JAFFE"] = collect_jaffe_pool(cfg.JAFFE_DIR)

    # 2. CK+
    print(f"\n  Scanning CK+: {cfg.CKP_DIR}")
    pools["CK+"] = collect_pool(cfg.CKP_DIR, max_per_class=999)

    # 3. RAF-DB  (compound split lives under RAF_DB_DIR/compound)
    #    Basic split lives under RAF_DB_DIR/basic or RAF_DB_DIR/train
    raf_basic_candidates = [
        os.path.join(cfg.RAF_DB_DIR, "basic"),
        os.path.join(cfg.RAF_DB_DIR, "train"),
        os.path.join(cfg.RAF_DB_DIR, "Image", "aligned"),
        cfg.RAF_DB_DIR,
    ]
    raf_basic_root = next(
        (p for p in raf_basic_candidates if os.path.exists(p)),
        cfg.RAF_DB_DIR
    )
    print(f"\n  Scanning RAF-DB basic: {raf_basic_root}")
    pools["RAF-DB"] = collect_pool(raf_basic_root, max_per_class=500)

    # 4. AffectNet
    print(f"\n  Scanning AffectNet: {cfg.AFFECTNET_DIR}")
    pools["AffectNet"] = collect_pool(cfg.AFFECTNET_DIR, max_per_class=500)

    # 5. FER-2013
    fer_pool = FER2013Pool(split="train")
    pools["FER-2013"] = fer_pool.by_class

    return pools


# ── Run all evaluations ───────────────────────────────────────────────────────
def run_all_dataset_evaluations(model, ckpt_name="stage3_best.pth",
                                 samples_per_class=100):
    """
    Main entry point. Call this cell after full_evaluation().
    Loads the best checkpoint, builds per-dataset pools,
    evaluates on each, and saves a summary JSON + bar chart.
    """
    # Load checkpoint
    load_ckpt(model, ckpt_name)
    model.eval()

    os.makedirs(cfg.EVAL_DIR, exist_ok=True)

    print(f"\n{'='*65}")
    print(f"  PER-DATASET EVALUATION  (ckpt: {ckpt_name})")
    print(f"  samples_per_class = {samples_per_class}")
    print(f"{'='*65}")

    pools   = build_all_pools()
    all_res = []

    for ds_name, by_class in pools.items():
        total_pool = sum(len(v) for v in by_class.values())
        if total_pool == 0:
            print(f"\n  ⚠  {ds_name}: pool is empty — check dataset path in cfg.")
            continue
        res = evaluate_dataset(
            model, by_class, ds_name,
            samples_per_class=samples_per_class
        )
        if res is not None:
            all_res.append(res)

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n\n{'='*65}")
    print(f"  SUMMARY — Compound Emotion Accuracy per Dataset")
    print(f"{'='*65}")
    print(f"  {'Dataset':<15} {'Acc':>8} {'W-F1':>8} {'M-F1':>8} {'N':>7}")
    print(f"  {'─'*50}")
    for r in all_res:
        print(f"  {r['dataset']:<15} "
              f"{r['accuracy']:>7.2f}% "
              f"{r['weighted_f1']:>7.2f}% "
              f"{r['macro_f1']:>7.2f}% "
              f"{r['n_samples']:>7}")

    # ── Bar chart ─────────────────────────────────────────────────────────────
    names = [r["dataset"]   for r in all_res]
    accs  = [r["accuracy"]  for r in all_res]
    f1s   = [r["weighted_f1"] for r in all_res]

    x = np.arange(len(names))
    w = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - w/2, accs, w, label="Accuracy",    color="#2E86AB")
    bars2 = ax.bar(x + w/2, f1s,  w, label="Weighted F1", color="#E84855", alpha=0.85)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{bar.get_height():.1f}%",
                ha="center", va="bottom", fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{bar.get_height():.1f}%",
                ha="center", va="bottom", fontsize=8)

    ax.axhline(65, color="green",  linestyle="--", linewidth=1,
               label="Target: 65%")
    ax.axhline(63.84, color="gray", linestyle=":",  linewidth=1,
               label="Val baseline: 63.84%")

    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=10)
    ax.set_ylabel("Score (%)")
    ax.set_title("Per-Dataset Compound Emotion Recognition",
                 fontweight="bold", fontsize=13)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    chart_path = os.path.join(cfg.EVAL_DIR, "per_dataset_summary.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  Bar chart saved → {chart_path}")

    # ── Save JSON ─────────────────────────────────────────────────────────────
    json_path = os.path.join(cfg.EVAL_DIR, "per_dataset_results.json")
    with open(json_path, "w") as f:
        json.dump(all_res, f, indent=2)
    print(f"  JSON saved     → {json_path}")
    print("\n  Download all files from the Kaggle Output tab.")

    return all_res


# ── RUN ───────────────────────────────────────────────────────────────────────
# samples_per_class: how many synthetic compound samples to generate per class
# per dataset. 100 is fast; bump to 180 for more stable estimates.
per_dataset_results = run_all_dataset_evaluations(
    model,
    ckpt_name="stage3_best.pth",
    samples_per_class=100,
)

  Loaded stage3_best.pth — compound_acc=64.34%

  PER-DATASET EVALUATION  (ckpt: stage3_best.pth)
  samples_per_class = 100
  JAFFE subfolders found: ['surprise', 'fear', 'angry', 'neutral', 'sad', 'disgust', 'happy']
    surprise → class 5 (Surprise): 0 imgs
    fear → class 2 (Fear): 0 imgs
    angry → class 0 (Angry): 0 imgs
    neutral → class 6 (Neutral): 0 imgs
    sad → class 4 (Sad): 0 imgs
    disgust → class 1 (Disgust): 0 imgs
    happy → class 3 (Happy): 0 imgs
  JAFFE total: 0 images

  Scanning CK+: /kaggle/input/datasets/prajwalm2213/dataset/CK
  Scanning: /kaggle/input/datasets/prajwalm2213/dataset/CK
  Found subfolders: ['anger', 'contempt', 'disgust', 'fear', 'happy', 'sadness', 'surprise']
    -> Matched 'anger' to class 0 (135 images)
    -> Matched 'disgust' to class 1 (177 images)
    -> Matched 'fear' to class 2 (75 images)
    -> Matched 'happy' to class 3 (207 images)
    -> Matched 'sadness' to class 4 (84 images)
    -> Matched 'surprise' to class 5 (249 imag

In [12]:
# ── Per-Dataset Evaluation Cell ───────────────────────────────────────────────
# Evaluates stage3_best.pth on each source dataset separately:
#   JAFFE, CK+, RAF-DB, AffectNet, FER-2013
#
# How it works:
#   Each dataset contains BASIC emotion images.
#   We synthesise compound images on-the-fly (same splice logic as training)
#   and evaluate the compound classifier on them.
#   This tells you how well each data source supports compound recognition.
# ─────────────────────────────────────────────────────────────────────────────

import os, random, json
import numpy as np
import torch
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns


# ── JAFFE pool ────────────────────────────────────────────────────────────────
# JAFFE folders are named like "AN", "DI", "FE", "HA", "NE", "SA", "SU"
JAFFE_FOLDER_MAP = {
    "an": 0, "di": 1, "fe": 2, "ha": 3,
    "ne": 6, "sa": 4, "su": 5,
}

def collect_jaffe_pool(root):
    """
    JAFFE images are PGM files named like 'KA.AN1.39.pgm'.
    The emotion code is the 2nd dot-segment (AN, DI, FE, HA, NE, SA, SU).
    """
    by_class = {i: [] for i in range(7)}
    if not os.path.exists(root):
        print(f"  ⚠  JAFFE not found: {root}")
        return by_class

    # Try subfolder structure first
    subfolders = [d for d in os.listdir(root)
                  if os.path.isdir(os.path.join(root, d))]
    if subfolders:
        print(f"  JAFFE subfolders found: {subfolders}")
        for folder in subfolders:
            idx = JAFFE_FOLDER_MAP.get(folder.lower()[:2])
            if idx is None:
                continue
            class_dir = os.path.join(root, folder)
            paths = [os.path.join(class_dir, f)
                     for f in os.listdir(class_dir)
                     if f.lower().endswith((".pgm", ".jpg", ".jpeg", ".png"))]
            by_class[idx].extend(paths)
            print(f"    {folder} → class {idx} ({cfg.BASIC_CLASSES[idx]}): "
                  f"{len(paths)} imgs")
    else:
        # Flat directory — parse emotion from filename
        print(f"  JAFFE flat directory, parsing filenames...")
        for fname in os.listdir(root):
            if not fname.lower().endswith((".pgm", ".jpg", ".jpeg", ".png")):
                continue
            parts = fname.split(".")
            if len(parts) >= 2:
                emo_code = parts[1][:2].lower()
                idx = JAFFE_FOLDER_MAP.get(emo_code)
                if idx is not None:
                    by_class[idx].append(os.path.join(root, fname))

    total = sum(len(v) for v in by_class.values())
    print(f"  JAFFE total: {total} images")
    for i, name in enumerate(cfg.BASIC_CLASSES):
        if by_class[i]:
            print(f"    {name:<10}: {len(by_class[i])}")
    return by_class


# ── Synthetic eval dataset (same splice as training) ─────────────────────────
class SyntheticEvalDataset(Dataset):
    """
    Builds N compound samples per class from a basic-emotion pool.
    Uses the same upper/lower splice as SyntheticCompoundDataset in training.
    """
    def __init__(self, by_class, samples_per_class=100, transform=None):
        self.transform = transform
        self.samples   = self._generate(by_class, samples_per_class)

    def _generate(self, by_class, n):
        samples = []
        for compound_idx, (e1, e2) in cfg.COMPOUND_TO_BASIC.items():
            pool1 = by_class.get(e1, [])
            pool2 = by_class.get(e2, [])
            if len(pool1) < 2 or len(pool2) < 2:
                print(f"    ⚠  {cfg.COMPOUND_CLASSES[compound_idx]}: "
                      f"skipped (e1={cfg.BASIC_CLASSES[e1]}:{len(pool1)}, "
                      f"e2={cfg.BASIC_CLASSES[e2]}:{len(pool2)})")
                continue
            actual_n = min(n, len(pool1), len(pool2))
            for _ in range(actual_n):
                samples.append((random.choice(pool1),
                                 random.choice(pool2),
                                 compound_idx))
        return samples

    def _load(self, path):
        img = cv2.imread(path)
        if img is None:
            img = np.array(Image.open(path).convert("RGB"))
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return cv2.resize(img, (cfg.IMAGE_SIZE, cfg.IMAGE_SIZE))

    def _splice(self, img1, img2):
        h     = img1.shape[0]
        alpha = random.uniform(0.45, 0.55)
        boundary = int(h * alpha)
        decay = 10
        result = img1.copy().astype(np.float32)
        result[boundary:] = img2[boundary:].astype(np.float32)
        for d in range(decay):
            row = boundary - decay // 2 + d
            if 0 <= row < h:
                t = d / decay
                result[row] = ((1 - t) * img1[row].astype(np.float32)
                               + t * img2[row].astype(np.float32))
        return np.clip(result, 0, 255).astype(np.uint8)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        p1, p2, label = self.samples[idx]
        mixed = self._splice(self._load(p1), self._load(p2))
        if self.transform:
            mixed = self.transform(image=mixed)["image"]
        return mixed, torch.tensor(label, dtype=torch.long)


# ── Evaluate one dataset ──────────────────────────────────────────────────────
def evaluate_dataset(model, by_class, dataset_name,
                     samples_per_class=100, batch_size=32):
    """
    Synthesises compound images from `by_class` pool and runs inference.
    Returns a results dict.
    """
    print(f"\n{'─'*60}")
    print(f"  Evaluating: {dataset_name}")
    print(f"{'─'*60}")

    eval_ds = SyntheticEvalDataset(
        by_class,
        samples_per_class=samples_per_class,
        transform=get_val_transforms()
    )

    if len(eval_ds) == 0:
        print(f"  ✗ No samples generated for {dataset_name} — skipping.")
        return None

    print(f"  Generated {len(eval_ds)} synthetic compound samples")

    loader = DataLoader(eval_ds, batch_size=batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(cfg.DEVICE)
            logits, _ = model(imgs)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    preds  = np.array(all_preds)
    labels = np.array(all_labels)

    # Only evaluate classes that actually have samples
    present_classes = sorted(set(labels.tolist()))
    present_names   = [cfg.COMPOUND_CLASSES[i] for i in present_classes]

    acc  = accuracy_score(labels, preds) * 100
    f1_w = f1_score(labels, preds, average="weighted", zero_division=0) * 100
    f1_m = f1_score(labels, preds, average="macro",    zero_division=0) * 100

    print(f"\n  {'Metric':<20} {'Value':>8}")
    print(f"  {'─'*30}")
    print(f"  {'Accuracy':<20} {acc:>7.2f}%")
    print(f"  {'Weighted F1':<20} {f1_w:>7.2f}%")
    print(f"  {'Macro F1':<20} {f1_m:>7.2f}%")
    print(f"  {'Samples evaluated':<20} {len(preds):>8}")

    report = classification_report(
        labels, preds,
        target_names=present_names,
        labels=present_classes,
        digits=4, zero_division=0
    )
    print(f"\n{report}")

    # Per-class accuracy
    cm  = confusion_matrix(labels, preds, labels=present_classes)
    pca = cm.diagonal() / (cm.sum(axis=1) + 1e-9) * 100
    print("  Per-class Accuracy:")
    for name, a in zip(present_names, pca):
        bar = "█" * int(a / 5)
        print(f"    {name:<25} {a:6.2f}%  {bar}")

    # Confusion matrix plot
    short = [n.replace("ily", "").replace("fully", "").replace("rily", "")
             for n in present_names]
    fig, ax = plt.subplots(figsize=(max(8, len(present_classes)),
                                    max(6, len(present_classes) - 1)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=short, yticklabels=short,
                linewidths=0.5, ax=ax)
    ax.set_title(f"Confusion Matrix — {dataset_name}",
                 fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.tight_layout()
    safe_name = dataset_name.replace(" ", "_").replace("/", "-").replace("+", "plus")
    cm_path   = os.path.join(cfg.EVAL_DIR, f"cm_{safe_name}.png")
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  Confusion matrix → {cm_path}")

    return {
        "dataset":          dataset_name,
        "n_samples":        int(len(preds)),
        "accuracy":         round(acc, 2),
        "weighted_f1":      round(f1_w, 2),
        "macro_f1":         round(f1_m, 2),
        "per_class_accuracy": {
            name: round(float(a), 2)
            for name, a in zip(present_names, pca)
        },
        "present_classes":  present_classes,
    }


# ── Build all per-dataset pools ───────────────────────────────────────────────
def build_all_pools():
    pools = {}

    # 1. JAFFE
    pools["JAFFE"] = collect_jaffe_pool(cfg.JAFFE_DIR)

    # 2. CK+
    print(f"\n  Scanning CK+: {cfg.CKP_DIR}")
    pools["CK+"] = collect_pool(cfg.CKP_DIR, max_per_class=999)

    # 3. RAF-DB  (compound split lives under RAF_DB_DIR/compound)
    #    Basic split lives under RAF_DB_DIR/basic or RAF_DB_DIR/train
    raf_basic_candidates = [
        os.path.join(cfg.RAF_DB_DIR, "basic"),
        os.path.join(cfg.RAF_DB_DIR, "train"),
        os.path.join(cfg.RAF_DB_DIR, "Image", "aligned"),
        cfg.RAF_DB_DIR,
    ]
    raf_basic_root = next(
        (p for p in raf_basic_candidates if os.path.exists(p)),
        cfg.RAF_DB_DIR
    )
    print(f"\n  Scanning RAF-DB basic: {raf_basic_root}")
    pools["RAF-DB"] = collect_pool(raf_basic_root, max_per_class=500)

    # 4. AffectNet
    print(f"\n  Scanning AffectNet: {cfg.AFFECTNET_DIR}")
    pools["AffectNet"] = collect_pool(cfg.AFFECTNET_DIR, max_per_class=500)

    # 5. FER-2013
    fer_pool = FER2013Pool(split="train")
    pools["FER-2013"] = fer_pool.by_class

    return pools


# ── Run all evaluations ───────────────────────────────────────────────────────
def run_all_dataset_evaluations(model, ckpt_name="stage3_best.pth",
                                 samples_per_class=100):
    """
    Main entry point. Call this cell after full_evaluation().
    Loads the best checkpoint, builds per-dataset pools,
    evaluates on each, and saves a summary JSON + bar chart.
    """
    # Load checkpoint
    load_ckpt(model, ckpt_name)
    model.eval()

    os.makedirs(cfg.EVAL_DIR, exist_ok=True)

    print(f"\n{'='*65}")
    print(f"  PER-DATASET EVALUATION  (ckpt: {ckpt_name})")
    print(f"  samples_per_class = {samples_per_class}")
    print(f"{'='*65}")

    pools   = build_all_pools()
    all_res = []

    for ds_name, by_class in pools.items():
        total_pool = sum(len(v) for v in by_class.values())
        if total_pool == 0:
            print(f"\n  ⚠  {ds_name}: pool is empty — check dataset path in cfg.")
            continue
        res = evaluate_dataset(
            model, by_class, ds_name,
            samples_per_class=samples_per_class
        )
        if res is not None:
            all_res.append(res)

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n\n{'='*65}")
    print(f"  SUMMARY — Compound Emotion Accuracy per Dataset")
    print(f"{'='*65}")
    print(f"  {'Dataset':<15} {'Acc':>8} {'W-F1':>8} {'M-F1':>8} {'N':>7}")
    print(f"  {'─'*50}")
    for r in all_res:
        print(f"  {r['dataset']:<15} "
              f"{r['accuracy']:>7.2f}% "
              f"{r['weighted_f1']:>7.2f}% "
              f"{r['macro_f1']:>7.2f}% "
              f"{r['n_samples']:>7}")

    # ── Bar chart ─────────────────────────────────────────────────────────────
    names = [r["dataset"]   for r in all_res]
    accs  = [r["accuracy"]  for r in all_res]
    f1s   = [r["weighted_f1"] for r in all_res]

    x = np.arange(len(names))
    w = 0.35
    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - w/2, accs, w, label="Accuracy",    color="#2E86AB")
    bars2 = ax.bar(x + w/2, f1s,  w, label="Weighted F1", color="#E84855", alpha=0.85)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{bar.get_height():.1f}%",
                ha="center", va="bottom", fontsize=8)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5,
                f"{bar.get_height():.1f}%",
                ha="center", va="bottom", fontsize=8)

    ax.axhline(65, color="green",  linestyle="--", linewidth=1,
               label="Target: 65%")
    ax.axhline(63.84, color="gray", linestyle=":",  linewidth=1,
               label="Val baseline: 63.84%")

    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=10)
    ax.set_ylabel("Score (%)")
    ax.set_title("Per-Dataset Compound Emotion Recognition",
                 fontweight="bold", fontsize=13)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    chart_path = os.path.join(cfg.EVAL_DIR, "per_dataset_summary.png")
    plt.savefig(chart_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  Bar chart saved → {chart_path}")

    # ── Save JSON ─────────────────────────────────────────────────────────────
    json_path = os.path.join(cfg.EVAL_DIR, "per_dataset_results.json")
    with open(json_path, "w") as f:
        json.dump(all_res, f, indent=2)
    print(f"  JSON saved     → {json_path}")
    print("\n  Download all files from the Kaggle Output tab.")

    return all_res


# ── RUN ───────────────────────────────────────────────────────────────────────
# samples_per_class: how many synthetic compound samples to generate per class
# per dataset. 100 is fast; bump to 180 for more stable estimates.
per_dataset_results = run_all_dataset_evaluations(
    model,
    ckpt_name="stage3_best.pth",
    samples_per_class=100,
)

  Loaded stage3_best.pth — compound_acc=64.34%

  PER-DATASET EVALUATION  (ckpt: stage3_best.pth)
  samples_per_class = 100
  JAFFE subfolders found: ['surprise', 'fear', 'angry', 'neutral', 'sad', 'disgust', 'happy']
    surprise → class 5 (Surprise): 0 imgs
    fear → class 2 (Fear): 0 imgs
    angry → class 0 (Angry): 0 imgs
    neutral → class 6 (Neutral): 0 imgs
    sad → class 4 (Sad): 0 imgs
    disgust → class 1 (Disgust): 0 imgs
    happy → class 3 (Happy): 0 imgs
  JAFFE total: 0 images

  Scanning CK+: /kaggle/input/datasets/prajwalm2213/dataset/CK
  Scanning: /kaggle/input/datasets/prajwalm2213/dataset/CK
  Found subfolders: ['anger', 'contempt', 'disgust', 'fear', 'happy', 'sadness', 'surprise']
    -> Matched 'anger' to class 0 (135 images)
    -> Matched 'disgust' to class 1 (177 images)
    -> Matched 'fear' to class 2 (75 images)
    -> Matched 'happy' to class 3 (207 images)
    -> Matched 'sadness' to class 4 (84 images)
    -> Matched 'surprise' to class 5 (249 imag

In [13]:
# ── DEBUG CELL — run this and share output ────────────────────────────────────

import os

# JAFFE
jaffe_root = "/kaggle/input/datasets/prajwalm2213/dataset2/jaffe/jaffe"
print("=== JAFFE ===")
print("Root contents:", os.listdir(jaffe_root))
first_subfolder = os.listdir(jaffe_root)[0]
first_path = os.path.join(jaffe_root, first_subfolder)
if os.path.isdir(first_path):
    files = os.listdir(first_path)
    print(f"Inside '{first_subfolder}': {len(files)} files")
    print("First 5:", files[:5])
else:
    print(f"'{first_subfolder}' is a file, not a folder")

# AffectNet
aff_test = "/kaggle/input/datasets/prajwalm2213/dataset/Affectnet/archive (3)/Test"
print("\n=== AffectNet/Test ===")
subfolders = os.listdir(aff_test)
print("Subfolders:", subfolders)
if subfolders:
    first = os.path.join(aff_test, subfolders[0])
    if os.path.isdir(first):
        files = os.listdir(first)
        print(f"Inside '{subfolders[0]}': {len(files)} files")
        print("First 3:", files[:3])

# RAF-DB test
raf_test = "/kaggle/input/datasets/prajwalm2213/dataset3/RAF-DB/test"
print("\n=== RAF-DB/test ===")
print("Subfolders:", os.listdir(raf_test))

=== JAFFE ===
Root contents: ['surprise', 'fear', 'angry', 'neutral', 'sad', 'disgust', 'happy']
Inside 'surprise': 30 files
First 5: ['TM.SU3.189.tiff', 'KR.SU3.82.tiff', 'NM.SU3.103.tiff', 'MK.SU3.124.tiff', 'KR.SU2.81.tiff']

=== AffectNet/Test ===
Subfolders: ['surprise', 'fear', 'neutral', 'sad', 'disgust', 'Contempt', 'happy', 'Anger']
Inside 'surprise': 1920 files
First 3: ['image0023058.jpg', 'image0020104.jpg', 'image0019245.jpg']

=== RAF-DB/test ===
Subfolders: ['surprise', 'fear', 'neutral', 'sadness', 'disgust', 'Happiness', 'anger']


In [14]:
# ── Per-Dataset Evaluation v2 ─────────────────────────────────────────────────
# Fixes:
#   JAFFE    — added .tiff extension
#   AffectNet — point to /Test subfolder, handle capitalised folder names
#   RAF-DB   — added /test split + handle 'Happiness', 'sadness' variants
# ─────────────────────────────────────────────────────────────────────────────

import os, random, json
import numpy as np
import torch
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# ── Extended folder→class map (covers all capitalisation variants) ────────────
FOLDER_TO_BASIC_EXT = {
    # anger
    "anger": 0, "angry": 0, "an": 0,
    # disgust
    "disgust": 1, "disgusted": 1, "di": 1,
    # fear
    "fear": 2, "fearful": 2, "fe": 2,
    # happy
    "happy": 3, "happiness": 3, "ha": 3,
    # sad
    "sad": 4, "sadness": 4, "sa": 4,
    # surprise
    "surprise": 5, "surprised": 5, "su": 5,
    # neutral
    "neutral": 6, "ne": 6,
    # AffectNet numeric
    "0": 6, "1": 3, "2": 4, "3": 5, "4": 2, "5": 1, "6": 0,
}

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".pgm")


def collect_pool_ext(root_dir, max_per_class=999):
    """
    Like collect_pool() but:
    - supports .tiff / .tif / .pgm
    - case-insensitive folder matching
    - no per-class cap by default (pass max_per_class to limit)
    """
    by_class = {i: [] for i in range(7)}
    if not os.path.exists(root_dir):
        print(f"  ⚠  Not found: {root_dir}")
        return by_class

    for folder in sorted(os.listdir(root_dir)):
        idx = FOLDER_TO_BASIC_EXT.get(folder.lower().strip())
        if idx is None:
            continue
        class_dir = os.path.join(root_dir, folder)
        if not os.path.isdir(class_dir):
            continue
        paths = [os.path.join(class_dir, f)
                 for f in os.listdir(class_dir)
                 if f.lower().endswith(IMG_EXTS)]
        random.shuffle(paths)
        by_class[idx].extend(paths[:max_per_class])
        print(f"    '{folder}' → class {idx} "
              f"({cfg.BASIC_CLASSES[idx]}): {len(by_class[idx])} imgs")

    total = sum(len(v) for v in by_class.values())
    print(f"  Total: {total} images")
    return by_class


# ── Synthetic eval dataset ────────────────────────────────────────────────────
class SyntheticEvalDataset(Dataset):
    def __init__(self, by_class, samples_per_class=100, transform=None):
        self.transform = transform
        self.samples   = self._generate(by_class, samples_per_class)

    def _generate(self, by_class, n):
        samples = []
        for compound_idx, (e1, e2) in cfg.COMPOUND_TO_BASIC.items():
            pool1 = by_class.get(e1, [])
            pool2 = by_class.get(e2, [])
            if len(pool1) < 2 or len(pool2) < 2:
                print(f"    ⚠  {cfg.COMPOUND_CLASSES[compound_idx]}: "
                      f"skipped (e1={cfg.BASIC_CLASSES[e1]}:{len(pool1)}, "
                      f"e2={cfg.BASIC_CLASSES[e2]}:{len(pool2)})")
                continue
            actual_n = min(n, len(pool1), len(pool2))
            for _ in range(actual_n):
                samples.append((random.choice(pool1),
                                 random.choice(pool2),
                                 compound_idx))
        return samples

    def _load(self, path):
        img = cv2.imread(path)
        if img is None:
            img = np.array(Image.open(path).convert("RGB"))
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return cv2.resize(img, (cfg.IMAGE_SIZE, cfg.IMAGE_SIZE))

    def _splice(self, img1, img2):
        h        = img1.shape[0]
        boundary = int(h * random.uniform(0.45, 0.55))
        decay    = 10
        result   = img1.copy().astype(np.float32)
        result[boundary:] = img2[boundary:].astype(np.float32)
        for d in range(decay):
            row = boundary - decay // 2 + d
            if 0 <= row < h:
                t = d / decay
                result[row] = ((1 - t) * img1[row].astype(np.float32)
                               + t      * img2[row].astype(np.float32))
        return np.clip(result, 0, 255).astype(np.uint8)

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        p1, p2, label = self.samples[idx]
        mixed = self._splice(self._load(p1), self._load(p2))
        if self.transform:
            mixed = self.transform(image=mixed)["image"]
        return mixed, torch.tensor(label, dtype=torch.long)


# ── Core evaluator ────────────────────────────────────────────────────────────
def evaluate_dataset(model, by_class, dataset_name,
                     samples_per_class=100, batch_size=32):
    print(f"\n{'─'*60}")
    print(f"  Evaluating: {dataset_name}")
    print(f"{'─'*60}")

    ds = SyntheticEvalDataset(by_class, samples_per_class,
                               transform=get_val_transforms())
    if len(ds) == 0:
        print(f"  ✗ No samples — skipping.")
        return None

    print(f"  Generated {len(ds)} synthetic compound samples")

    loader = DataLoader(ds, batch_size=batch_size,
                        shuffle=False, num_workers=2, pin_memory=True)

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            logits, _ = model(imgs.to(cfg.DEVICE))
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    preds  = np.array(all_preds)
    labels = np.array(all_labels)

    present   = sorted(set(labels.tolist()))
    p_names   = [cfg.COMPOUND_CLASSES[i] for i in present]

    acc  = accuracy_score(labels, preds) * 100
    f1_w = f1_score(labels, preds, average="weighted", zero_division=0) * 100
    f1_m = f1_score(labels, preds, average="macro",    zero_division=0) * 100

    print(f"\n  {'Metric':<20} {'Value':>8}")
    print(f"  {'─'*30}")
    print(f"  {'Accuracy':<20} {acc:>7.2f}%")
    print(f"  {'Weighted F1':<20} {f1_w:>7.2f}%")
    print(f"  {'Macro F1':<20} {f1_m:>7.2f}%")
    print(f"  {'Samples':<20} {len(preds):>8}")

    report = classification_report(
        labels, preds, target_names=p_names,
        labels=present, digits=4, zero_division=0)
    print(f"\n{report}")

    cm  = confusion_matrix(labels, preds, labels=present)
    pca = cm.diagonal() / (cm.sum(axis=1) + 1e-9) * 100
    print("  Per-class Accuracy:")
    for name, a in zip(p_names, pca):
        bar = "█" * int(a / 5)
        print(f"    {name:<25} {a:6.2f}%  {bar}")

    # Confusion matrix plot
    short   = [n.replace("ily","").replace("fully","").replace("rily","")
                for n in p_names]
    sz      = max(8, len(present))
    fig, ax = plt.subplots(figsize=(sz, sz - 1))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=short, yticklabels=short,
                linewidths=0.5, ax=ax)
    ax.set_title(f"Confusion Matrix — {dataset_name}",
                 fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.tight_layout()
    safe = dataset_name.replace(" ","_").replace("/","-").replace("+","plus")
    cm_path = os.path.join(cfg.EVAL_DIR, f"cm_{safe}.png")
    plt.savefig(cm_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  Confusion matrix → {cm_path}")

    return {
        "dataset":            dataset_name,
        "n_samples":          int(len(preds)),
        "accuracy":           round(acc,  2),
        "weighted_f1":        round(f1_w, 2),
        "macro_f1":           round(f1_m, 2),
        "per_class_accuracy": {n: round(float(a), 2)
                                for n, a in zip(p_names, pca)},
        "present_classes":    present,
    }


# ── Build all pools (v2 — correct paths & extensions) ─────────────────────────
def build_all_pools_v2():
    pools = {}

    # 1. JAFFE — .tiff files inside named subfolders
    print("\n=== JAFFE ===")
    print(f"  Path: {cfg.JAFFE_DIR}")
    pools["JAFFE"] = collect_pool_ext(cfg.JAFFE_DIR, max_per_class=999)

    # 2. CK+ — unchanged, already working
    print("\n=== CK+ ===")
    print(f"  Path: {cfg.CKP_DIR}")
    pools["CK+"] = collect_pool_ext(cfg.CKP_DIR, max_per_class=999)

    # 3. RAF-DB train
    raf_train = os.path.join(cfg.RAF_DB_DIR, "train")
    print(f"\n=== RAF-DB (train) ===")
    print(f"  Path: {raf_train}")
    pools["RAF-DB (train)"] = collect_pool_ext(raf_train, max_per_class=500)

    # 4. RAF-DB test  ← new
    raf_test = os.path.join(cfg.RAF_DB_DIR, "test")
    print(f"\n=== RAF-DB (test) ===")
    print(f"  Path: {raf_test}")
    pools["RAF-DB (test)"] = collect_pool_ext(raf_test, max_per_class=500)

    # 5. AffectNet — point to /Test subfolder
    aff_test = os.path.join(cfg.AFFECTNET_DIR, "Test")
    print(f"\n=== AffectNet (Test) ===")
    print(f"  Path: {aff_test}")
    pools["AffectNet"] = collect_pool_ext(aff_test, max_per_class=500)

    # 6. FER-2013
    print("\n=== FER-2013 ===")
    fer_pool = FER2013Pool(split="train")
    pools["FER-2013"] = fer_pool.by_class

    return pools


# ── Main ──────────────────────────────────────────────────────────────────────
def run_all_dataset_evaluations_v2(model, ckpt_name="stage3_best.pth",
                                    samples_per_class=100):
    load_ckpt(model, ckpt_name)
    model.eval()
    os.makedirs(cfg.EVAL_DIR, exist_ok=True)

    print(f"\n{'='*65}")
    print(f"  PER-DATASET EVALUATION v2  (ckpt: {ckpt_name})")
    print(f"  samples_per_class = {samples_per_class}")
    print(f"{'='*65}")

    pools   = build_all_pools_v2()
    all_res = []

    for ds_name, by_class in pools.items():
        if sum(len(v) for v in by_class.values()) == 0:
            print(f"\n  ⚠  {ds_name}: pool empty — skipping.")
            continue
        res = evaluate_dataset(model, by_class, ds_name,
                               samples_per_class=samples_per_class)
        if res:
            all_res.append(res)

    # ── Summary table ─────────────────────────────────────────────────────────
    print(f"\n\n{'='*65}")
    print(f"  SUMMARY — Compound Accuracy per Dataset")
    print(f"{'='*65}")
    print(f"  {'Dataset':<22} {'Acc':>8} {'W-F1':>8} {'M-F1':>8} {'N':>7}")
    print(f"  {'─'*57}")
    for r in all_res:
        flag = " ✅" if r["accuracy"] >= 65 else (" 🟡" if r["accuracy"] >= 55 else " 🔴")
        print(f"  {r['dataset']:<22} "
              f"{r['accuracy']:>7.2f}% "
              f"{r['weighted_f1']:>7.2f}% "
              f"{r['macro_f1']:>7.2f}% "
              f"{r['n_samples']:>7}{flag}")

    # ── Bar chart ─────────────────────────────────────────────────────────────
    names = [r["dataset"]     for r in all_res]
    accs  = [r["accuracy"]    for r in all_res]
    f1s   = [r["weighted_f1"] for r in all_res]

    x = np.arange(len(names))
    w = 0.35
    fig, ax = plt.subplots(figsize=(max(10, len(names) * 1.8), 6))
    b1 = ax.bar(x - w/2, accs, w, label="Accuracy",    color="#2E86AB")
    b2 = ax.bar(x + w/2, f1s,  w, label="Weighted F1", color="#E84855",
                alpha=0.85)

    for bar in list(b1) + list(b2):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.8,
                f"{bar.get_height():.1f}%",
                ha="center", va="bottom", fontsize=8)

    ax.axhline(65,    color="green", linestyle="--", linewidth=1.2,
               label="Target 65%")
    ax.axhline(63.84, color="gray",  linestyle=":",  linewidth=1,
               label="Val baseline 63.84%")

    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=9, rotation=15, ha="right")
    ax.set_ylabel("Score (%)")
    ax.set_ylim(0, 108)
    ax.set_title("Per-Dataset Compound Emotion Recognition",
                 fontweight="bold", fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    chart = os.path.join(cfg.EVAL_DIR, "per_dataset_summary_v2.png")
    plt.savefig(chart, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n  Bar chart → {chart}")

    # ── Save JSON ──────────────────────────────────────────────────────────────
    out = os.path.join(cfg.EVAL_DIR, "per_dataset_results_v2.json")
    with open(out, "w") as f:
        json.dump(all_res, f, indent=2)
    print(f"  JSON      → {out}")
    print("\n  Download from Kaggle Output tab.")
    return all_res


# ── RUN ───────────────────────────────────────────────────────────────────────
per_dataset_results = run_all_dataset_evaluations_v2(
    model,
    ckpt_name="stage3_best.pth",
    samples_per_class=100,
)

  Loaded stage3_best.pth — compound_acc=64.34%

  PER-DATASET EVALUATION v2  (ckpt: stage3_best.pth)
  samples_per_class = 100

=== JAFFE ===
  Path: /kaggle/input/datasets/prajwalm2213/dataset2/jaffe/jaffe
    'angry' → class 0 (Angry): 30 imgs
    'disgust' → class 1 (Disgust): 29 imgs
    'fear' → class 2 (Fear): 32 imgs
    'happy' → class 3 (Happy): 31 imgs
    'neutral' → class 6 (Neutral): 30 imgs
    'sad' → class 4 (Sad): 31 imgs
    'surprise' → class 5 (Surprise): 30 imgs
  Total: 213 images

=== CK+ ===
  Path: /kaggle/input/datasets/prajwalm2213/dataset/CK
    'anger' → class 0 (Angry): 135 imgs
    'disgust' → class 1 (Disgust): 177 imgs
    'fear' → class 2 (Fear): 75 imgs
    'happy' → class 3 (Happy): 207 imgs
    'sadness' → class 4 (Sad): 84 imgs
    'surprise' → class 5 (Surprise): 249 imgs
  Total: 927 images

=== RAF-DB (train) ===
  Path: /kaggle/input/datasets/prajwalm2213/dataset3/RAF-DB/train
    'anger' → class 0 (Angry): 500 imgs
    'disgust' → class 1 (Dis